# Subtitle (Whisper) + CLAP trên video gốc → Volume `keyframes-btc`

`docs/triet/ke-hoach-offline-ingest.md` §4.4, §4.5, §5.6 · config `config/ingest_shards_batch1_btc_video.yaml`

Chạy sau `02-keyframes-btc-import.ipynb`. **Cùng cách chia người** với 02 và 03 — đổi `MEMBER` là ra
đúng phần việc quen thuộc.

| Tầng | Model | Ra artifact |
|---|---|---|
| `n2_subtitles` | `faster-whisper large-v3` | `subtitle_segments` (thô) + `subtitles` (theo keyframe) |
| `n3_audio` | `laion/clap-htsat-unfused` | `audio_features` |

### Không tải gói video nào

Gói `Videos_*.zip` lớn nhất là 11,97 GB trên đĩa ~20 GB. Notebook này lấy **từng video một bằng
HTTP Range** (`mfe.offline.source`): mọi mp4 trong các gói đó đều STORED nên cắt ra là dùng được
ngay. Đỉnh dùng đĩa ~325 MB thay vì 12 GB, và phiên chết chỉ mất một video chứ không mất một gói.

Hệ quả đáng chú ý: vì không tải theo gói nữa, **video nằm trong gói nào không còn quan trọng**, nên
lượt này giữ nguyên cách chia của lượt nhập keyframe thay vì chia lại theo gói video. Điều đó là
bắt buộc chứ không phải tiện tay — xem đầu file config.

### Hai tầng phải đi cùng phiên

`StageContext.audio_path()` tách wav **một lần** rồi N2 và N3 dùng chung. Chạy N3 ở phiên khác là
giải mã lại toàn bộ audio cho không.

### N2 là tầng duy nhất không phải làm lại khi DAKE chốt

`subtitle_segments` là timestamp thô của video — không dính bộ keyframe. Khi tham số DAKE chốt và
bundle DAKE ra đời, file này dùng lại nguyên vẹn, chỉ phải suy lại ánh xạ sang keyframe (rẻ).
Whisper không phải chạy lần hai. Mọi tầng khác trong bundle này đánh khoá theo từng keyframe nên
thuộc riêng bộ BTC.

### Điều kiện dừng của N2

Tầng **ném lỗi** nếu >30% keyframe không nhận được subtitle trong khi Whisper vẫn ra segment (§4.5):
đó thường là sai cửa sổ hoặc sai kênh audio, và chạy tiếp sẽ chôn nó vào manifest. Video đó bị đánh
LỖI và bỏ qua, các video khác chạy tiếp.


## Step 0a — repo + gói

In [ ]:
MEMBER      = "triet"          # <-- ĐỔI THÀNH TÊN BẠN
SHARDS      = "config/ingest_shards_batch1_btc_video.yaml"
TRANSPORT   = "modal"
VOLUME      = "keyframes-btc"  # CÙNG volume với notebook 02 và 03
VOL_VERSION = 2
KEEP_MODELS = True             # False = thả model sau mỗi video (chậm hơn nhiều)
REPO_URL    = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF    = "feature/triet/structure"

import os, subprocess, sys
from pathlib import Path

REPO = Path("/kaggle/input/mfe-repo")
if not REPO.is_dir():
    REPO = Path("/kaggle/working/mfe")
    if not REPO.is_dir():
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_REF, url, str(REPO)],
                       check=True, capture_output=True)   # token không lọt vào log
sys.path.insert(0, str(REPO / "src"))

req = REPO / "notebooks" / "requirements-kaggle.lock.txt"
if not req.is_file():
    req = REPO / "notebooks" / "requirements-kaggle-btc-video.txt"
print(f"cài từ {req.name}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)

## Step 0b — Modal, phân công, việc đã xong

In [ ]:
import logging, yaml
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from kaggle_secrets import UserSecretsClient
from mfe.offline.bundle import BundleLayout, BundleWriter, StageStatus, load_shard_plan
from mfe.offline.source import fetch_video, load_catalogue
from mfe.offline.stages import STAGES, StageContext, read_keyframes, unload_models
from mfe.offline.transport import build_transport

secrets = UserSecretsClient()
os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")

RAW       = yaml.safe_load((REPO / SHARDS).read_text(encoding="utf-8"))
plan      = load_shard_plan(REPO / SHARDS)
layout    = BundleLayout(plan.bundle)
transport = build_transport(TRANSPORT, volume=VOLUME,
                            local_root="/kaggle/working/bundle_local",
                            version=VOL_VERSION)
WORK   = Path("/kaggle/working/scratch"); WORK.mkdir(parents=True, exist_ok=True)
ZIPS   = Path("/kaggle/working/zips");    ZIPS.mkdir(parents=True, exist_ok=True)
VIDEOS = Path("/kaggle/working/videos");  VIDEOS.mkdir(parents=True, exist_ok=True)
writer = BundleWriter(transport, layout, MEMBER, WORK)

assert plan.keyframe_set == "btc"
STAGE_LIST = plan.stages
PARAMS     = RAW.get("params", {})
EXTRAS     = RAW["extras"]

# AI LÀM VIDEO NÀO (theo gói keyframe) vs VIDEO NẰM Ở ĐÂU (theo gói video).
CATALOGUE = load_catalogue(REPO / RAW["video_catalogue"])
MY_VIDEOS = plan.videos_for(MEMBER)
status    = StageStatus.load(transport, layout, MEMBER)

no_entry = [v for v in MY_VIDEOS if v not in CATALOGUE]
missing_import = [v for v in MY_VIDEOS if not status.is_done("n0_import_btc", v)]
todo = [(s, v) for v in MY_VIDEOS for s in STAGE_LIST if not status.is_done(s, v)]

print(f"bundle  : {plan.bundle}  ->  volume {VOLUME} (v{VOL_VERSION})")
print(f"người   : {MEMBER}")
print(f"video   : {len(MY_VIDEOS)} (tổng {len(plan.videos)})")
print(f"tầng    : {', '.join(STAGE_LIST)}")
print(f"còn lại : {len(todo)} việc / {len(MY_VIDEOS) * len(STAGE_LIST)}")
if no_entry:
    raise SystemExit(f"{len(no_entry)} video khong co entry Range: {no_entry[:5]}")
if missing_import:
    print()
    print(f"CHU Y: {len(missing_import)} video CHUA nhap bang notebook 02: "
          f"{', '.join(missing_import[:6])}{' …' if len(missing_import) > 6 else ''}")
    print("   Chay xong 02 truoc — notebook nay se BO QUA chung.")

In [ ]:
import shutil, zipfile

def free_gb():
    return shutil.disk_usage("/kaggle/working").free / 2**30

def download(url, name):
    """Tải một gói nhỏ, có resume."""
    target = ZIPS / name
    if target.is_file():
        print(f"    {name}: đã có ({target.stat().st_size / 2**20:.0f} MB)")
        return target
    print(f"    tải {name} …")
    # --read-timeout la thu quan trong nhat: mot ket noi bi bop xuong ~400 B/s
    # van la ket noi "song", nen khong co no thi wget ngoi doi vo han thay vi
    # noi lai. -c nen thu lai khong bao gio tai lai tu dau.
    subprocess.run(
        ["wget", "-q", "--show-progress", "-c",
         "--timeout=30", "--read-timeout=60", "--tries=0", "--waitretry=10",
         "--retry-connrefused", "-O", str(target), url],
        check=True,
    )
    return target

def extract_one_file(zip_path, inner_suffix, dest):
    with zipfile.ZipFile(zip_path) as archive:
        match = [n for n in archive.namelist() if n.endswith(inner_suffix)]
        if not match:
            raise FileNotFoundError(f"{inner_suffix} khong co trong {zip_path.name}")
        dest.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(match[0]) as src, dest.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    return dest

# `map-keyframes` (~1 MB) lấy từ gói của BTC, KHÔNG kéo từ Volume.
#
# Ảnh keyframe thì KHÔNG cần ở lượt này: N2 và N3 chỉ đọc `pts_time`/`frame_idx`
# trong csv để đặt cửa sổ. `read_keyframes` có dựng đường dẫn ảnh, nhưng không
# tầng nào ở đây mở chúng — nên thư mục ảnh rỗng là đúng, không phải thiếu.
print("gói dùng chung:")
MAPS_ZIP = download(EXTRAS["map_keyframes"], "map-keyframes-aic25-b1.zip")
print(f"đĩa trống: {free_gb():.1f} GB")

## Step 1 — vòng lặp

Mỗi video: lấy bằng Range → tách wav → N2 → N3 → đẩy → xoá. Model nằm trong cache nên chỉ nạp một
lần cho cả phiên (Whisper large-v3 ~3 GB + CLAP ~0,6 GB ≈ 3,6 GB / 15 GB của T4).

`merge=True` khi ghi manifest: bundle này do **ba notebook** cùng đổ (02 nhập, 03 ảnh, 04 đây), cả
ba cùng ghi `part-manifest/{video}.json`. Không gộp thì mỗi lượt xoá sạch artifact của lượt trước,
mà marker vẫn báo "xong" nên không lần chạy nào phục hồi lại.

In [ ]:
import time, traceback

failures = []
started = time.perf_counter()
done_count = 0

pending = [v for v in MY_VIDEOS
           if status.is_done("n0_import_btc", v)
           and any(not status.is_done(s, v) for s in STAGE_LIST)]
print(f"phiên này sẽ chạy {len(pending)} video")

for video_id in pending:
    stages = [s for s in STAGE_LIST if not status.is_done(s, video_id)]
    entry = CATALOGUE[video_id]
    video_path = VIDEOS / f"{video_id}.mp4"
    wav_path = WORK / f"{video_id}.wav"
    try:
        print()
        print(f"  {video_id}  {entry.size / 2**20:.0f} MB tu {entry.zip_name}  "
              f"({len(stages)} tầng)")
        fetch_video(entry, video_path)
        map_csv = extract_one_file(MAPS_ZIP, f"/{video_id}.csv",
                                   WORK / "map-keyframes" / f"{video_id}.csv")
        keyframes = read_keyframes(map_csv, WORK / "keyframes" / video_id)

        ctx = StageContext(video_id, video_path, keyframes, writer, layout, WORK)
        markers = []
        for stage in stages:
            ctx.params = dict(PARAMS.get(stage, {}))
            marker = STAGES[stage](ctx)
            markers.append(marker)
            status.done.add((stage, video_id))
            print(f"      {stage:<16} {marker.get('rows', '-'):>6} dòng  "
                  f"{marker.get('elapsed_s')}s")

        writer.write_part_manifest(video_id, markers, keyframes=len(keyframes), merge=True)
        done_count += 1
        rate = (time.perf_counter() - started) / done_count
        print(f"  [{done_count:>3}/{len(pending)}] xong | còn ~"
              f"{(len(pending) - done_count) * rate / 3600:.1f}h")
    except Exception as err:
        failures.append((video_id, repr(err)))
        print(f"  [LỖI] {video_id}: {err}")
        traceback.print_exc(limit=3)
    finally:
        video_path.unlink(missing_ok=True)
        wav_path.unlink(missing_ok=True)
        (WORK / "map-keyframes" / f"{video_id}.csv").unlink(missing_ok=True)
        if not KEEP_MODELS:
            unload_models()

print()
print("=" * 72)
print(f"phiên này: {done_count} video, {len(failures)} lỗi, "
      f"{(time.perf_counter() - started) / 3600:.1f}h")
for video_id, err in failures:
    print(f"  {video_id}: {err}")

## Step 2 — chốt phiên

In [ ]:
unload_models()

fresh = StageStatus.load(transport, layout, MEMBER)
left  = [(s, v) for v in MY_VIDEOS for s in STAGE_LIST if not fresh.is_done(s, v)]

total = len(MY_VIDEOS) * len(STAGE_LIST)
print(f"{MEMBER}: {total - len(left)}/{total} việc đã lên {VOLUME}")
for stage in STAGE_LIST:
    n = sum(1 for v in MY_VIDEOS if fresh.is_done(stage, v))
    print(f"  {stage:<16} {n:>4}/{len(MY_VIDEOS)} video")

if left:
    print()
    print(f"còn {len(left)} việc — bấm Run All lại, nó chạy tiếp từ chỗ đứt")
else:
    print()
    print("XONG phần của bạn. Báo Lead.")
    print(f"  python scripts/verify_bundle.py --shards {SHARDS} --member {MEMBER}")